# 01 | Data Loading and Inspection

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Use simple, reusable Python functions to load real IMU and model-derived strain data, inspect arrays and participant labels, then visualise representative signals. No training is performed.

**Dataset:** Authors' public BoneNet walking/running derivative, not the entire Essex Figshare archive. Subject-row matching and target units require further independent validation.

## 1. Environment and imports

The main functions live in .py files. This cell adds the local source folders to the Python path, so **no custom hmi package or editable installation is needed**. Install standard dependencies once from a terminal using: python -m pip install -r requirements.txt.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / 'requirements.txt').is_file() or not (root / 'code/python/src/core/data_loading.py').is_file():
    if root.parent == root:
        raise RuntimeError('Open this notebook from within human-movement-intelligence.')
    root = root.parent

sys.path.insert(0, str(root / 'code' / 'python' / 'src'))
sys.path.insert(0, str(root / 'studies' / 'biomechanical_load_estimation' / 'src'))

import matplotlib.pyplot as plt
from IPython.display import display

from core.data_loading import load_data
from core.data_inspection import inspect_data, plot_signal
from bonenet_data import ensure_bonenet_files, load_bonenet_data

print('Project:', root)
print('Python:', sys.executable)
print('Data-loading functions: ready')

## 2. Prepare the data source

ensure_bonenet_files() downloads Data.mat and labels.csv **once** from the authors' public repository, if missing. They are stored under data/raw/ and ignored by Git.

In [ ]:
data_dir = root / 'data' / 'raw' / 'bonenet_reference'
ensure_bonenet_files(data_dir)
print('Source data folder:', data_dir)

## 3. Read the source with a general reader

load_data() understands CSV, Excel, text and standard MAT files. It does **not** assume that a particular variable is an IMU signal.

In [ ]:
labels_table = load_data(data_dir / 'labels.csv')
source_mat = load_data(data_dir / 'Data.mat')
print('MAT variables:', [key for key in source_mat if not key.startswith('__')])
label_report = inspect_data(labels_table, name='BoneNet label table')
print('Distinct subject IDs:', labels_table['SubjID'].nunique())

## 4. Convert the source layout into numeric arrays

The **BoneNet-specific** adapter assembles the x/y/z IMU channels and e1/e3 output arrays. This step is not part of the general file reader.

In [ ]:
X, y, labels, channel_names, target_names = load_bonenet_data(data_dir)
input_report = inspect_data(X, name='IMU input: N x channels x time')
target_report = inspect_data(y, name='FE target: N x time x outputs')
print('Example channels:', channel_names[:6])
print('Target channels:', target_names)

## 5. Plot actual signal values

These are examples from the source arrays, not simulated figures. The original physical units and the correctness of row-order label associations have **not** been independently established.

In [ ]:
example_record = 0
fig1 = plot_signal(X[example_record, :3, :], title='BoneNet: first three IMU channels',
                   channel_names=channel_names[:3])
plt.show()

fig2 = plot_signal(y[example_record].T, title='Model-derived FE strain (source scale)',
                   channel_names=target_names)
plt.show()

## 6. Quality gate and interpretation

This notebook demonstrates loading, profiling and plotting, **not** data leakage control, validated biomechanical ground truth or model evaluation.

In [ ]:
assert X.shape[0] == y.shape[0] == len(labels)
assert X.ndim == 3 and y.ndim == 3
print('Structural data loading: PASSED')
print('Independent subject-to-row alignment: NOT VERIFIED')
print('Model training performed: NO')